# StylePops — Kombin + A/B Üretimi (Colab GPU)

**Mac'te çalıştırma.** Bu notebook kombinleri GPU'da üretir (~3–5 dk).

## Ön hazırlık (Mac'te bir kez)
```bash
cd StylePops_Modules
python scripts/pack_colab_combo_bundle.py
```
`outputs/colab_combo_bundle.zip` dosyasını Google Drive'a yükle:
`MyDrive/StylePops_colab/colab_combo_bundle.zip`

## Colab
1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**

In [ ]:
import torch
assert torch.cuda.is_available(), 'Runtime → GPU (T4) seç!'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip install -q torch torchvision transformers accelerate peft pillow joblib lightgbm

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

PROJECT = Path('/content/StylePops_Modules')
os.chdir('/content')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
!git clone -q https://github.com/valueisinvalid/StylePops_Modules.git
os.chdir(PROJECT)
print('Repo:', PROJECT)

In [ ]:
from google.colab import drive
from pathlib import Path
import zipfile, shutil

drive.mount('/content/drive')
BUNDLE_ZIP = Path('/content/drive/MyDrive/StylePops_colab/colab_combo_bundle.zip')
assert BUNDLE_ZIP.exists(), f'Yok: {BUNDLE_ZIP}\nMac: python scripts/pack_colab_combo_bundle.py → Drive\'a yükle'

with zipfile.ZipFile(BUNDLE_ZIP) as zf:
    zf.extractall('/content')

SRC = Path('/content/stylepops_colab')
for rel in ('data/visual/garments_production.json', 'data/visual/inventory_registry.json'):
    shutil.copy2(SRC / rel, PROJECT / rel)
shutil.copytree(SRC / 'data' / 'lookups', PROJECT / 'data' / 'lookups', dirs_exist_ok=True)
for folder in ('data/assets/garments', 'data/assets/fashion_product'):
    src_dir = SRC / folder
    if src_dir.exists():
        shutil.copytree(src_dir, PROJECT / folder, dirs_exist_ok=True)
lora_src = SRC / 'outputs' / 'fashionclip_lora'
if lora_src.exists():
    shutil.copytree(lora_src, PROJECT / 'outputs' / 'fashionclip_lora', dirs_exist_ok=True)
    print('LoRA ✓')

import json
n = json.loads((PROJECT / 'data/visual/garments_production.json').read_text())['count']
print(f'Gardırop: {n} parça yüklendi')

In [ ]:
import subprocess, sys
from pathlib import Path

Path('data/assets/combos').mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, '-u', 'scripts/generate_visual_combinations.py', '--ab-pairs', '200']
print(' '.join(cmd))
subprocess.run(cmd, check=True, cwd='/content/StylePops_Modules')

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

root = Path('/content/StylePops_Modules')
out = Path('/content/stylepops_results.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in ('data/visual/combinations_visual.csv', 'data/visual/ab_pairs.csv'):
        zf.write(root / name, Path(name).name)

combos_zip = Path('/content/stylepops_combos.zip')
!cd /content/StylePops_Modules/data/assets/combos && zip -qr {combos_zip} AB*.jpg VC*.jpg

print('İndiriliyor… Mac\'te:')
print('  combinations_visual.csv + ab_pairs.csv → data/visual/')
print('  AB*.jpg VC*.jpg → data/assets/combos/')
files.download(str(out))
files.download(str(combos_zip))